# 02 Cluster Analysis and Streaming Simulation

This notebook demonstrates a file-based Spark Structured Streaming workflow using the credit card fraud transaction dataset.  
The goal is to simulate real-time ingestion, apply reusable transformations, and monitor a live fraud aggregation as new batch files arrive.

In [1]:
from pathlib import Path
import threading
import shutil
import time

from IPython.display import clear_output
from pyspark.sql.functions import col

from project.streaming import prepare_stream_batches, run_stream_producer
from project.config import load_config
from project.spark_session import create_spark
from project.schemas import TRANSACTION_SCHEMA
from project.transform import prepare_transactions

## Load configuration and define paths

We load the project configuration and define the core paths used in this notebook:
- the raw input dataset
- the folder of prepared micro-batch CSV files
- the streaming input folder that Spark will watch

In [2]:
repo_root = Path.cwd().parent
cfg = load_config(repo_root / "configs" / "local.yaml")

raw_csv = repo_root / cfg.paths.raw
batch_dir = repo_root / cfg.paths.stream_batches
input_dir = repo_root / cfg.paths.stream_input

## Define a reset utility for the streaming environment

To make the notebook reproducible, we define a helper function that clears both:
- the batch folder
- the streaming input folder

This ensures each run starts fresh and Spark sees all incoming files as new data.

In [3]:
def reset_stream_environment(batch_dir, input_dir):
    """
    Fully reset streaming environment:
    - Deletes batch files
    - Deletes streaming input files
    - Recreates clean directories
    """
    if batch_dir.exists():
        shutil.rmtree(batch_dir)
    batch_dir.mkdir(parents=True, exist_ok=True)

    if input_dir.exists():
        shutil.rmtree(input_dir)
    input_dir.mkdir(parents=True, exist_ok=True)

    print("Streaming environment reset:")
    print(f"  batch_dir: {batch_dir}")
    print(f"  input_dir: {input_dir}")

## Reset the batch and streaming folders

We clear any files from previous runs so the streaming simulation behaves consistently each time the notebook is executed.

In [4]:
reset_stream_environment(batch_dir, input_dir)

Streaming environment reset:
  batch_dir: /home/jovyan/work/data/stream/batches
  input_dir: /home/jovyan/work/data/stream/input


## Split the raw dataset into micro-batches

We divide the large raw CSV into smaller CSV files.  
These smaller files simulate streaming data when they are copied into the watched input folder over time.

In [5]:
prepare_stream_batches(
    source_csv=raw_csv,
    batch_dir=batch_dir
)

Creating batch files...
Batch creation complete.


## Start the Spark session

We initialize a local Spark session using the project configuration and reduce Spark log noise for easier notebook output.

In [6]:
spark = create_spark(cfg)
spark.sparkContext.setLogLevel("WARN")

print("Spark session created.")

Spark session created.


## Read the streaming input source

Spark Structured Streaming watches the input folder and reads new CSV files as they arrive.  
The transaction schema is applied explicitly to ensure consistent column types.

In [7]:
stream_df = (
    spark.readStream
         .option("header", True)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
         .option("dateFormat", "yyyy-MM-dd")
         .schema(TRANSACTION_SCHEMA)
         .csv(str(input_dir))
)

## Apply reusable transformation logic

We use the shared transformation pipeline from `transform.py` to clean and enrich the streaming data.

This step adds analytical features such as:
- transaction date
- transaction hour
- transaction month
- transaction day of week
- customer age
- log-transformed transaction amount

In [8]:
stream_df = prepare_transactions(stream_df)

## Create a live fraud aggregation

We filter the stream to fraudulent transactions only and aggregate by transaction hour.  
This produces a running count of fraud events by hour as new data arrives.

In [9]:
fraud_by_hour = (
    stream_df
    .filter(col("is_fraud") == 1)
    .groupBy("event_hour")
    .count()
)

## Start the streaming query

We write the aggregation to an in-memory Spark table.  
This allows us to query the live results repeatedly from within the notebook.

In [10]:
query = (
    fraud_by_hour.writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("fraud_by_hour_mem")
    .start()
)

print("Streaming query started.")

Streaming query started.


## Simulate real-time data arrival

A background producer thread gradually copies prepared batch files into the streaming input folder.

This mimics a live data feed while Spark continues to monitor and process new files.

In [11]:
stop_event = threading.Event()

producer_thread = threading.Thread(
    target=run_stream_producer,
    kwargs={
        "batch_dir": batch_dir,
        "streaming_dir": input_dir,
        "timeout_seconds": 1800, #30 minutes
        "min_delay": 1,
        "max_delay": 6,
        "min_files_per_push": 1,
        "max_files_per_push": 10,
        "loop_batches": True,
        "stop_event": stop_event,
        "clear_streaming_dir": False,
    },
    daemon=True
)

producer_thread.start()
print("Producer running in background.")

Producer running in background.
Starting stream producer...
Sent 1 files | Total: 1
Sent 10 files | Total: 11
Sent 4 files | Total: 15
Sent 10 files | Total: 25
Sent 1 files | Total: 26
Sent 5 files | Total: 31
Sent 6 files | Total: 37
Sent 9 files | Total: 46
Sent 2 files | Total: 48
Sent 4 files | Total: 52
Sent 3 files | Total: 55
Sent 9 files | Total: 64
Sent 4 files | Total: 68
Sent 9 files | Total: 77
Sent 3 files | Total: 80
Sent 2 files | Total: 82
Sent 7 files | Total: 89
Sent 9 files | Total: 98
Sent 2 files | Total: 100
Sent 6 files | Total: 106
Sent 3 files | Total: 109
Sent 2 files | Total: 111
Sent 4 files | Total: 115
Sent 10 files | Total: 125
Sent 8 files | Total: 133


## Monitor the live aggregation

This loop repeatedly queries the in-memory result table to show how the fraud-by-hour aggregation changes as new files arrive.

The loop gives the notebook a real-time updates while the streaming query runs in the background.

In [12]:
start_time = time.time()
current_time = time.time()

while current_time - start_time <= 120:
    clear_output(wait=True)

    print("Live fraud count by hour")
    print(f"Query active: {query.isActive}")
    print(f"Elapsed: {int(current_time - start_time)} seconds\n")

    spark.sql("""
        SELECT *
        FROM fraud_by_hour_mem
        ORDER BY event_hour
    """).show(50, truncate=False)

    current_time = time.time()
    time.sleep(5)

Live fraud count by hour
Query active: True
Elapsed: 116 seconds

+----------+-----+
|event_hour|count|
+----------+-----+
|0         |805  |
|1         |854  |
|2         |774  |
|3         |760  |
|4         |59   |
|5         |75   |
|6         |52   |
|7         |74   |
|8         |60   |
|9         |68   |
|10        |53   |
|11        |52   |
|12        |79   |
|13        |100  |
|14        |109  |
|15        |105  |
|16        |100  |
|17        |96   |
|18        |97   |
|19        |102  |
|20        |79   |
|21        |91   |
|22        |2391 |
|23        |2397 |
+----------+-----+

Sent 7 files | Total: 324
Sent 8 files | Total: 332
Sent 3 files | Total: 335


## Inspect query status

We can check whether the streaming query is still active and review its latest execution progress.

In [15]:
print("isActive:", query.isActive)
print("status:", query.status)

isActive: False
status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}


## Stop the streaming query

When the demonstration is complete, we explicitly stop the query to release resources and end the streaming job cleanly.

In [14]:
for q in spark.streams.active:
    q.stop()

stop_event.set()
producer_thread.join(timeout=5)

print("All active queries stopped and producer stopped.")

Stop signal received during delay. Producer stopping.
All active queries stopped and producer stopped.
